# Collaboration and Competition

---

In this notebook, you will learn how to use the Unity ML-Agents environment for the third project of the [Deep Reinforcement Learning Nanodegree](https://www.udacity.com/course/deep-reinforcement-learning-nanodegree--nd893) program.

### 1. Start the Environment

We begin by importing the necessary packages.  If the code cell below returns an error, please revisit the project instructions to double-check that you have installed [Unity ML-Agents](https://github.com/Unity-Technologies/ml-agents/blob/master/docs/Installation.md) and [NumPy](http://www.numpy.org/).

In [1]:
from unityagents import UnityEnvironment
import numpy as np

Next, we will start the environment!  **_Before running the code cell below_**, change the `file_name` parameter to match the location of the Unity environment that you downloaded.

- **Mac**: `"path/to/Tennis.app"`
- **Windows** (x86): `"path/to/Tennis_Windows_x86/Tennis.exe"`
- **Windows** (x86_64): `"path/to/Tennis_Windows_x86_64/Tennis.exe"`
- **Linux** (x86): `"path/to/Tennis_Linux/Tennis.x86"`
- **Linux** (x86_64): `"path/to/Tennis_Linux/Tennis.x86_64"`
- **Linux** (x86, headless): `"path/to/Tennis_Linux_NoVis/Tennis.x86"`
- **Linux** (x86_64, headless): `"path/to/Tennis_Linux_NoVis/Tennis.x86_64"`

For instance, if you are using a Mac, then you downloaded `Tennis.app`.  If this file is in the same folder as the notebook, then the line below should appear as follows:
```
env = UnityEnvironment(file_name="Tennis.app")
```

In [2]:
env = UnityEnvironment(file_name="Tennis_Windows_x86_64/Tennis.exe")

INFO:unityagents:
'Academy' started successfully!
Unity Academy name: Academy
        Number of Brains: 1
        Number of External Brains : 1
        Lesson number : 0
        Reset Parameters :
		
Unity brain name: TennisBrain
        Number of Visual Observations (per agent): 0
        Vector Observation space type: continuous
        Vector Observation space size (per agent): 8
        Number of stacked Vector Observation: 3
        Vector Action space type: continuous
        Vector Action space size (per agent): 2
        Vector Action descriptions: , 


Environments contain **_brains_** which are responsible for deciding the actions of their associated agents. Here we check for the first brain available, and set it as the default brain we will be controlling from Python.

In [3]:
# get the default brain
brain_name = env.brain_names[0]
brain = env.brains[brain_name]

### 2. Examine the State and Action Spaces

In this environment, two agents control rackets to bounce a ball over a net. If an agent hits the ball over the net, it receives a reward of +0.1.  If an agent lets a ball hit the ground or hits the ball out of bounds, it receives a reward of -0.01.  Thus, the goal of each agent is to keep the ball in play.

The observation space consists of 8 variables corresponding to the position and velocity of the ball and racket. Two continuous actions are available, corresponding to movement toward (or away from) the net, and jumping. 

Run the code cell below to print some information about the environment.

In [4]:
# reset the environment
env_info = env.reset(train_mode=True)[brain_name]

# number of agents 
num_agents = len(env_info.agents)
print('Number of agents:', num_agents)

# size of each action
action_size = brain.vector_action_space_size
print('Size of each action:', action_size)

# examine the state space 
states = env_info.vector_observations
state_size = states.shape[1]
print('There are {} agents. Each observes a state with length: {}'.format(states.shape[0], state_size))
print('The state for the first agent looks like:', states[0])

Number of agents: 2
Size of each action: 2
There are 2 agents. Each observes a state with length: 24
The state for the first agent looks like: [ 0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.         -6.65278625 -1.5
 -0.          0.          6.83172083  6.         -0.          0.        ]


### 3. Take Random Actions in the Environment

In the next code cell, you will learn how to use the Python API to control the agents and receive feedback from the environment.

Once this cell is executed, you will watch the agents' performance, if they select actions at random with each time step.  A window should pop up that allows you to observe the agents.

Of course, as part of the project, you'll have to change the code so that the agents are able to use their experiences to gradually choose better actions when interacting with the environment!

In [6]:
for i in range(1, 6):                                      # play game for 5 episodes
    env_info = env.reset(train_mode=False)[brain_name]     # reset the environment    
    states = env_info.vector_observations                  # get the current state (for each agent)
    scores = np.zeros(num_agents)                          # initialize the score (for each agent)
    while True:
        actions = np.random.randn(num_agents, action_size) # select an action (for each agent)
        actions = np.clip(actions, -1, 1)                  # all actions between -1 and 1
        env_info = env.step(actions)[brain_name]           # send all actions to tne environment
        next_states = env_info.vector_observations         # get next state (for each agent)
        rewards = env_info.rewards                         # get reward (for each agent)
        dones = env_info.local_done                        # see if episode finished
        scores += env_info.rewards                         # update the score (for each agent)
        states = next_states                               # roll over states to next time step
        if np.any(dones):                                  # exit loop if episode finished
            break
    print('Score (max over agents) from episode {}: {}'.format(i, np.max(scores)))

Score (max over agents) from episode 1: 0.10000000149011612
Score (max over agents) from episode 2: 0.0
Score (max over agents) from episode 3: 0.0
Score (max over agents) from episode 4: 0.0
Score (max over agents) from episode 5: 0.0


When finished, you can close the environment.

In [7]:
env.close()

### 4. It's Your Turn!

Now it's your turn to train your own agent to solve the environment!  When training the environment, set `train_mode=True`, so that the line for resetting the environment looks like the following:
```python
env_info = env.reset(train_mode=True)[brain_name]
```

In [1]:
# train.py
import os
import numpy as np
from collections import deque
import torch

from unityagents import UnityEnvironment
from agent import MASAC

# ---------------------------
# Tuned hyperparameters (change here)
# ---------------------------
CONFIG = {
    "env_path": "Tennis_Windows_x86_64/Tennis.exe",               # None -> try Editor; else set path e.g. "./Tennis.exe"
    "n_episodes": 3000,
    "max_t": 1000,
    "print_every": 100,             # print & checkpoint interval (episodes)
    # MASAC params
    "lr_actor": 3e-4,
    "lr_critic": 3e-4,
    "gamma": 0.99,
    "tau": 0.005,
    "alpha": 0.1,                   # tuned for faster learning; lower -> less random
    "hidden_actor": 128,
    "hidden_critic": 256,
    "batch_size": 256,
    "buffer_size": int(1e6),
    "update_steps": 2,              # updates per env step
    "seed": 0
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def save_checkpoint(agent, episode, avg_score, name="latest"):
    os.makedirs("checkpoints", exist_ok=True)
    path = os.path.join("checkpoints", f"{name}.pth")
    payload = {
        "episode": episode,
        "avg_score": float(avg_score),
        "critic_state": agent.critic.state_dict(),
        "critic_target_state": agent.critic_target.state_dict(),
        "actors_state": [a.state_dict() for a in agent.actors],
        "config": CONFIG
    }
    torch.save(payload, path)

def get_env(env_path=None):
    if env_path is None:
        env = UnityEnvironment(file_name=None)  # connect to editor
    else:
        env = UnityEnvironment(file_name=env_path)
    return env

def train():
    env = get_env(CONFIG["env_path"])
    brain_name = env.brain_names[0]
    brain = env.brains[brain_name]

    env_info = env.reset(train_mode=True)[brain_name]
    num_agents = len(env_info.agents)
    state_size = env_info.vector_observations.shape[1]
    action_size = brain.vector_action_space_size

    print(f"Agents: {num_agents}, state_size: {state_size}, action_size: {action_size}")
    agent = MASAC(
        num_agents=num_agents,
        state_size=state_size,
        action_size=action_size,
        lr_actor=CONFIG["lr_actor"],
        lr_critic=CONFIG["lr_critic"],
        gamma=CONFIG["gamma"],
        tau=CONFIG["tau"],
        alpha=CONFIG["alpha"],
        hidden_actor=CONFIG["hidden_actor"],
        hidden_critic=CONFIG["hidden_critic"],
        batch_size=CONFIG["batch_size"],
        buffer_size=CONFIG["buffer_size"],
        update_steps=CONFIG["update_steps"],
        seed=CONFIG["seed"]
    )

    scores_deque = deque(maxlen=100)
    best_avg = -np.inf

    for ep in range(1, CONFIG["n_episodes"] + 1):
        env_info = env.reset(train_mode=True)[brain_name]
        states = env_info.vector_observations      # shape (N, state_size)
        agent_done = False
        score = np.zeros(num_agents)

        for t in range(CONFIG["max_t"]):
            actions = agent.act(states)            # (N, A)
            env_info = env.step(actions)[brain_name]

            next_states = env_info.vector_observations
            rewards = np.array(env_info.rewards)
            dones = np.array(env_info.local_done, dtype=np.uint8)

            agent.step(states, actions, rewards, next_states, dones)
            states = next_states
            score += rewards

            if np.any(dones):
                break

        scores_deque.append(np.max(score))

        # periodic printing/checkpoint every print_every episodes
        if ep % CONFIG["print_every"] == 0:
            avg = np.mean(scores_deque)
            print(f"Ep {ep}\tAvg Score (last 100): {avg:.3f}")


            # if improvement over previous best: save 'best'
            if avg > best_avg:
                best_avg = avg
                save_checkpoint(agent, ep, avg, name="best")
                print(f"New BEST model saved! Avg: {avg:.3f}")

        # solved condition
        if np.mean(scores_deque) >= 0.5 and ep >= 100:
            print(f"Solved in {ep} episodes! Avg: {np.mean(scores_deque):.3f}")
            save_checkpoint(agent, ep, np.mean(scores_deque), name="solved")
            break

    env.close()

if __name__ == "__main__":
    train()


INFO:unityagents:
'Academy' started successfully!
Unity Academy name: Academy
        Number of Brains: 1
        Number of External Brains : 1
        Lesson number : 0
        Reset Parameters :
		
Unity brain name: TennisBrain
        Number of Visual Observations (per agent): 0
        Vector Observation space type: continuous
        Vector Observation space size (per agent): 8
        Number of stacked Vector Observation: 3
        Vector Action space type: continuous
        Vector Action space size (per agent): 2
        Vector Action descriptions: , 


Agents: 2, state_size: 24, action_size: 2
Ep 100	Avg Score (last 100): 0.012
New BEST model saved! Avg: 0.012
Ep 200	Avg Score (last 100): 0.020
New BEST model saved! Avg: 0.020
Ep 300	Avg Score (last 100): 0.026
New BEST model saved! Avg: 0.026
Ep 400	Avg Score (last 100): 0.027
New BEST model saved! Avg: 0.027
Ep 500	Avg Score (last 100): 0.029
New BEST model saved! Avg: 0.029
Ep 600	Avg Score (last 100): 0.030
New BEST model saved! Avg: 0.030
Ep 700	Avg Score (last 100): 0.038
New BEST model saved! Avg: 0.038
Ep 800	Avg Score (last 100): 0.037
Ep 900	Avg Score (last 100): 0.044
New BEST model saved! Avg: 0.044
Ep 1000	Avg Score (last 100): 0.034
Ep 1100	Avg Score (last 100): 0.059
New BEST model saved! Avg: 0.059
Ep 1200	Avg Score (last 100): 0.049
Ep 1300	Avg Score (last 100): 0.058
Ep 1400	Avg Score (last 100): 0.072
New BEST model saved! Avg: 0.072
Ep 1500	Avg Score (last 100): 0.060
Ep 1600	Avg Score (last 100): 0.074
New BEST model saved! Avg: 0.074
Ep 1700	Avg Score (last 100)